In [55]:
import pandas as pd
import numpy as np

BASE_PATH = "../data_collection/"
df = pd.read_csv(BASE_PATH + "amazon_delivery.csv")

In [56]:
df.isnull().sum()

Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather            91
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64

In [57]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         43739 non-null  object 
 1   Agent_Age        43739 non-null  int64  
 2   Agent_Rating     43685 non-null  float64
 3   Store_Latitude   43739 non-null  float64
 4   Store_Longitude  43739 non-null  float64
 5   Drop_Latitude    43739 non-null  float64
 6   Drop_Longitude   43739 non-null  float64
 7   Order_Date       43739 non-null  object 
 8   Order_Time       43739 non-null  object 
 9   Pickup_Time      43739 non-null  object 
 10  Weather          43648 non-null  object 
 11  Traffic          43739 non-null  object 
 12  Vehicle          43739 non-null  object 
 13  Area             43739 non-null  object 
 14  Delivery_Time    43739 non-null  int64  
 15  Category         43739 non-null  object 
dtypes: float64(5), int64(2), object(9)
memory usage: 5.3+ MB


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [58]:
df = df.drop("Order_ID", axis=1)

In [59]:
#Combine Order_Date and Order_Time
df["Order_Time"] = pd.to_datetime(
    df["Order_Date"].astype(str) + " " + df["Order_Time"].astype(str),
    errors="coerce"
)
df["Pickup_Time"] = pd.to_datetime(
    df["Order_Date"].astype(str) + " " + df["Pickup_Time"].astype(str),
    errors="coerce"
)

# clean Nan and nul value 
df = df.dropna(subset=["Order_Time"])
df = df.dropna(subset=["Pickup_Time"])

#rename the column
df.rename(columns={"Order_Time": "Order_DateTime"}, inplace=True)
df.rename(columns={"Pickup_Time": "Pickup_DateTime"}, inplace=True)
df.rename(columns={"Delivery_Time": "Delivery_Duration"}, inplace=True)

In [60]:
df = df.drop("Order_Date", axis=1)

In [61]:
# bad_rows = df[
#     pd.to_datetime(df["Order_DateTime"], format="%H:%M:%S", errors="coerce").isnull()
# ]

# bad_rows["Order_DateTime"].head(10)

In [62]:
df["Traffic"].unique()

array(['High ', 'Jam ', 'Low ', 'Medium '], dtype=object)

In [63]:
#clean white space
df["Traffic"] = df["Traffic"].str.strip()

In [64]:
df["Traffic"] = df["Traffic"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Jam": 3
})

In [65]:
df = pd.get_dummies(
    df,
    columns=["Weather", "Vehicle", "Category", "Area"],
    drop_first=True
)

In [66]:
list(df.columns)

['Agent_Age',
 'Agent_Rating',
 'Store_Latitude',
 'Store_Longitude',
 'Drop_Latitude',
 'Drop_Longitude',
 'Order_DateTime',
 'Pickup_DateTime',
 'Traffic',
 'Delivery_Duration',
 'Weather_Fog',
 'Weather_Sandstorms',
 'Weather_Stormy',
 'Weather_Sunny',
 'Weather_Windy',
 'Vehicle_scooter ',
 'Vehicle_van',
 'Category_Books',
 'Category_Clothing',
 'Category_Cosmetics',
 'Category_Electronics',
 'Category_Grocery',
 'Category_Home',
 'Category_Jewelry',
 'Category_Kitchen',
 'Category_Outdoors',
 'Category_Pet Supplies',
 'Category_Shoes',
 'Category_Skincare',
 'Category_Snacks',
 'Category_Sports',
 'Category_Toys',
 'Area_Other',
 'Area_Semi-Urban ',
 'Area_Urban ']

In [69]:
#Get Distance b/w store and drop location 
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

df["distance_km"] = haversine(
    df["Store_Latitude"], df["Store_Longitude"],
    df["Drop_Latitude"], df["Drop_Longitude"]
)

,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_DateTime,Pickup_DateTime,Traffic,Delivery_Duration,...,Category_Pet Supplies,Category_Shoes,Category_Skincare,Category_Snacks,Category_Sports,Category_Toys,Area_Other,Area_Semi-Urban,Area_Urban,distance_km
0,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19 11:30:00,2022-03-19 11:45:00,2,120,...,False,False,False,False,False,False,False,False,True,3.025149
1,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25 19:45:00,2022-03-25 19:50:00,3,165,...,False,False,False,False,False,False,False,False,False,20.183530
2,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19 08:30:00,2022-03-19 08:45:00,0,130,...,False,False,False,False,True,False,False,False,True,1.552758
3,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05 18:00:00,2022-04-05 18:10:00,1,105,...,False,False,False,False,False,False,False,False,False,7.790401
4,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26 13:30:00,2022-03-26 13:45:00,2,150,...,False,False,False,False,False,True,False,False,False,6.210138
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43734,30,4.8,26.902328,75.794257,26.912328,75.804257,2022-03-24 11:35:00,2022-03-24 11:45:00,2,160,...,False,False,False,False,False,False,False,False,False,1.489846
43735,21,4.6,0.000000,0.000000,0.070000,0.070000,2022-02-16 19:55:00,2022-02-16 20:10:00,3,180,...,False,False,False,False,False,False,False,False,False,11.007735
43736,30,4.9,13.022394,80.242439,13.052394,80.272439,2022-03-11 23:50:00,2022-03-11 00:05:00,0,80,...,False,False,False,False,False,False,False,False,False,4.657195
43737,20,4.7,11.001753,76.986241,11.041753,77.026241,2022-03-07 13:35:00,2022-03-07 13:40:00,2,130,...,False,False,False,False,False,False,False,False,False,6.232393


In [71]:
#Drop ["Store_Latitude", "Store_Longitude", "Drop_Latitude", "Drop_Longitude"]
df = df.drop(["Store_Latitude", "Store_Longitude", "Drop_Latitude", "Drop_Longitude"], axis=1)